# Encoder–decoder Transformer with RoPE for legal continuation

Six encoder and six decoder layers, MHA, full encoder self-attention, causal
decoder self-attention, encoder–decoder cross-attention, post-LayerNorm and ReLU
FFNs. RoPE rotates queries and keys in **encoder and decoder self-attention**.
Cross-attention has no rotary transformation. Embedding scaling and dropout are
preserved; additive sinusoidal positional encoding is removed. No nn.Transformer.

Architecture reference: https://arxiv.org/abs/1706.03762
RoPE reference: https://arxiv.org/abs/2104.09864


## 1. Hugging Face Login

In [1]:
%pip install -q huggingface_hub ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/150.2 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 143.4/150.2 kB 6.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 3.2 MB/s eta 0:00:00


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 3. Configuration

In [4]:
import os
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# You can use any of the 4 train files or all the files for pre-training the legal model
TRAIN_FILE = ("Phase_1_Pretraining/Train/train_acts.json",
             "Phase_1_Pretraining/Train/train_high_courts.json",
             "Phase_1_Pretraining/Train/train_supreme_court.json",
             "Phase_1_Pretraining/Train/train_tribunals.json")
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
VAL_TOKENS = 500_000
TRAIN_BLOCK_SIZE = 1024
SEED = 42
random.seed(SEED)

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 1024
TEMPERATURE = 0.8


# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase1_encdec_rope_2048_400m"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-transformer-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")
# Edit training settings here before running later cells.
TRAIN_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 16
TRAIN_STEPS = 80000  # Microbatches; 6,250 updates with accumulation 16.
WARMUP_UPDATES = 1000  # 16% of the planned 6,250 optimizer updates.
LR_FACTOR = 0.15       # Multiplier on the paper's learning-rate formula.
LABEL_SMOOTHING = 0.0  # Optimize token NLL directly.
TOP_K = 50
USE_TWO_GPUS = False  # Set True to use DataParallel during training.
# Optional: path to latest.pt created by THIS notebook; keep best.pt beside it.
# TRAIN_STEPS remains the total target when resuming.
RESUME_FROM = None


Configuration loaded.


## 4. Load Tokenizer

In [ ]:
from collections import defaultdict, Counter
import heapq


SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]


class BytelevelBPE:

    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.specials = SPECIALS.copy()
        self.byte_vocab = [str(i) for i in range(256)]
        self.merges = []
        self.merge_ranks = {}
        self.vocab = []
        self.stoi = {}
        self.itos = {}
        # Maps each BPE symbol to the bytes represented by it
        self.symbol_to_bytes = {}
    
    # 1. WORD -> UTF-8 BYTE SYMBOLS

    def word_to_symbols(self, word):
        byte_seq = word.encode("utf-8")
        return [str(b) for b in byte_seq] + ["</w>"]

    # 2. BUILD WORD FREQUENCIES

    def build_word_freqs(self, corpus):
        word_counter = Counter()
        print("Building word frequencies...")
        for text in corpus:
            if not isinstance(text, str):
                continue

            word_counter.update(text.split())
        print(
            f"Unique whitespace-separated words: "
            f"{len(word_counter):,}"
        )
        # Convert each unique word to byte symbols only once
        word_freqs = {}

        for word, freq in word_counter.items():
            symbols = tuple(self.word_to_symbols(word))
            word_freqs[symbols] = (
                word_freqs.get(symbols, 0) + freq
            )

        return word_freqs


    # 3. PAIRS OCCURRING IN ONE WORD

    @staticmethod
    def get_pairs(symbols):
        return {
            (symbols[i], symbols[i + 1])
            for i in range(len(symbols) - 1)
        }

    # 4. COUNT PAIR OCCURRENCES IN ONE WORD

    @staticmethod
    def pair_counts_in_word(symbols):
        counts = Counter()
        for i in range(len(symbols) - 1):
            counts[
                (symbols[i], symbols[i + 1])
            ] += 1

        return counts

    # 5. MERGE A PAIR INSIDE ONE WORD

    @staticmethod
    def merge_symbols(symbols, pair, merged_symbol):
        new_symbols = []
        i = 0
        while i < len(symbols):
            if (
                i < len(symbols) - 1
                and symbols[i] == pair[0]
                and symbols[i + 1] == pair[1]
            ):
                new_symbols.append(merged_symbol)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1

        return tuple(new_symbols)

    # 6. TRAIN OPTIMIZED BPE

    def train(self, corpus):
        word_freqs = self.build_word_freqs(corpus)
        vocab = (
            self.specials
            + self.byte_vocab
            + ["</w>"]
        )
        seen = set(vocab)
        for b in range(256):
            self.symbol_to_bytes[str(b)] = bytes([b])
        self.symbol_to_bytes["</w>"] = b""

        words = []
        frequencies = []
        for symbols, freq in word_freqs.items():
            words.append(symbols)
            frequencies.append(freq)
        del word_freqs

        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)

        print("Building initial pair statistics...")

        for word_id, symbols in enumerate(words):
            freq = frequencies[word_id]
            local_counts = self.pair_counts_in_word(symbols)
            for pair, count in local_counts.items():
                pair_counts[pair] += count * freq
                pair_to_words[pair].add(word_id)

        print(
            f"Initial unique pairs: "
            f"{len(pair_counts):,}"
        )

        # MAX HEAP
        #
        # Python has min-heap, so use negative frequency.
        #
        # We keep stale entries and validate them when popped.
        # This avoids rebuilding the heap after every merge.

        heap = []
        for pair, count in pair_counts.items():
            heapq.heappush(heap,(-count, pair))
        target_merges = max(0,self.vocab_size - len(vocab))
        print(f"Target merges: {target_merges:,}")
        while (len(vocab) < self.vocab_size and heap):
            while heap:
                neg_count, pair = heapq.heappop(heap)
                heap_count = -neg_count
                current_count = pair_counts.get(pair,0)
                # Ignore stale heap entries
                if (current_count == heap_count and current_count > 0):
                    break
            else:
                break

            max_freq = pair_counts.get(pair, 0)

            # Same stopping condition as before
            if max_freq <= 1:
                break

            # Create collision-free merged symbol
            merged_symbol = (f"<bpe_{len(self.merges)}>")

            # Byte representation of new symbol
            left_bytes = self.symbol_to_bytes[pair[0]]
            right_bytes = self.symbol_to_bytes[pair[1]]
            self.symbol_to_bytes[merged_symbol] = left_bytes + right_bytes

            self.merges.append(pair)
            self.merge_ranks[pair] = (len(self.merges) - 1,merged_symbol)
            if merged_symbol not in seen:
                seen.add(merged_symbol)
                vocab.append(merged_symbol)

            # ----------------------------------------------------
            # IMPORTANT OPTIMIZATION
            #
            # Only process words containing selected pair.
            # ----------------------------------------------------

            affected_word_ids = list(pair_to_words.get(pair, set()))
            changed_pairs = set()
            for word_id in affected_word_ids:
                old_symbols = words[word_id]
                freq = frequencies[word_id]
                # Pair may have disappeared because of an earlier
                # update involving this word
                if pair not in self.get_pairs(old_symbols):
                    continue
                
                # Statistics BEFORE merge
                old_counts = self.pair_counts_in_word(old_symbols)

                # Merge selected pair
                new_symbols = self.merge_symbols(old_symbols,pair,merged_symbol)
                if new_symbols == old_symbols:
                    continue

                # Statistics AFTER merge
                new_counts = self.pair_counts_in_word(new_symbols)

                # Save updated word
                words[word_id] = new_symbols

                all_pairs = (
                    set(old_counts)
                    | set(new_counts)
                )

                # Update only pairs affected by this word

                for p in all_pairs:

                    old_n = old_counts.get(p, 0)
                    new_n = new_counts.get(p, 0)
                    delta = (
                        new_n - old_n
                    ) * freq

                    if delta != 0:
                        pair_counts[p] += delta
                        changed_pairs.add(p)

                    # Update inverted index
                    if new_n > 0:
                        pair_to_words[p].add(word_id)
                    else:
                        pair_to_words[p].discard(word_id)

            # Selected pair should now be gone from affected words
            pair_to_words[pair].clear()

            # Push updated frequencies into heap
            for p in changed_pairs:
                count = pair_counts.get(p, 0)
                if count > 0:
                    heapq.heappush(heap,(-count, p))

        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.stoi = {
            symbol: i
            for i, symbol in enumerate(vocab)
        }

        self.itos = {
            i: symbol
            for symbol, i in self.stoi.items()
        }

        print("\nTokenizer training complete.")

        print(
            "Final vocabulary size:",
            self.vocab_size
        )

        print(
            "Number of merges:",
            len(self.merges)
        )

    # 7. APPLY BPE TO ONE WORD

    def apply_bpe(self, word):
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best_pair = None
            best_rank = float("inf")
            best_merged_symbol = None

            for i in range(len(symbols) - 1):
                pair = (symbols[i],symbols[i + 1])
                merge_info = self.merge_ranks.get(pair)
                if merge_info is not None:
                    rank, merged_symbol = merge_info
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = pair
                        best_merged_symbol = merged_symbol

            if best_pair is None:
                break

            symbols = list(
                self.merge_symbols(
                    tuple(symbols),
                    best_pair,
                    best_merged_symbol
                )
            )

        return symbols

    # 8. ENCODE

    def encode(self,text,add_bos=False,add_eos=False):
        ids = []
        if add_bos:
            ids.append(
                self.stoi["<s>"]
            )

        for word in text.split():
            symbols = self.apply_bpe(word)
            for symbol in symbols:
                ids.append(
                    self.stoi.get(
                        symbol,
                        self.stoi["<unk>"]
                    )
                )

        if add_eos:
            ids.append(
                self.stoi["</s>"]
            )

        return ids

    # 9. DECODE

    def decode(self, ids):

        words = []
        current_bytes = bytearray()

        for token_id in ids:
            symbol = self.itos.get(token_id)
            if symbol is None:
                continue
            if symbol in self.specials:
                continue

            # Determine whether this token ends a word
            ends_word = self._symbol_ends_word(symbol)
            byte_value = self.symbol_to_bytes.get(symbol, b"")
            current_bytes.extend(byte_value)

            if ends_word:
                try:
                    word = bytes(current_bytes).decode("utf-8")
                    words.append(word)
                except UnicodeDecodeError:
                    words.append("<unk>")
                current_bytes = bytearray()

        # Remaining bytes
        if current_bytes:
            try:
                words.append(bytes(current_bytes).decode("utf-8"))
            except UnicodeDecodeError:
                words.append("<unk>")

        return " ".join(words)

    # 10. CHECK WHETHER SYMBOL CONTAINS </w> AT END

    def _symbol_ends_word(self, symbol):
        if symbol == "</w>":
            return True
        # Find merge that created this symbol
        for pair, (_, merged) in self.merge_ranks.items():
            if merged == symbol:
                return self._symbol_ends_word(
                    pair[1]
                )

        return False

    # 11. VOCAB SIZE

    def get_vocab_size(self):
        return len(self.vocab)

## 5. Download Training Data From Hugging Face

In [6]:
from huggingface_hub import hf_hub_download

train_file_path_acts = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE[0],
    repo_type="dataset",
    token=HF_TOKEN
)

train_file_path_high_courts = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE[1],
    repo_type="dataset",
    token=HF_TOKEN
)

train_file_path_supreme_court = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE[2],
    repo_type="dataset",
    token=HF_TOKEN
)

train_file_path_tribunals = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE[3],
    repo_type="dataset",
    token=HF_TOKEN
)

print("Training file:")
print(train_file_path_acts)

print("Training file:")
print(train_file_path_high_courts)

print("Training file:")
print(train_file_path_supreme_court)

print("Training file:")
print(train_file_path_tribunals)

Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

Training file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json
Training file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json
Training file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json
Training file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json


In [7]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [8]:
source_paths = {
    "acts": train_file_path_acts,
    "high_courts": train_file_path_high_courts,
    "supreme_court": train_file_path_supreme_court,
    "tribunals": train_file_path_tribunals,
}

In [9]:
import os
import json
import random
import hashlib
import numpy as np

SOURCE_RATIOS = {
    "acts": 1,
    "high_courts": 2,
    "supreme_court": 3,
    "tribunals": 2,
}

TOKENIZER_WORD_BUDGET = 4_000_000
TRAIN_TOKEN_BUDGET = 400_000_000
VAL_TOKEN_BUDGET = 500_000  # 500,000 in your configuration

# Use your chosen document-level validation fraction.
VALIDATION_FRACTION = 0.05

def allocate_budget(total):
    ratio_sum = sum(SOURCE_RATIOS.values())

    budgets = {
        name: total * ratio // ratio_sum
        for name, ratio in SOURCE_RATIOS.items()
    }

    # Handle rounding for budgets not divisible by five.
    remainder = total - sum(budgets.values())
    for name in list(budgets)[:remainder]:
        budgets[name] += 1

    return budgets

word_budgets = allocate_budget(TOKENIZER_WORD_BUDGET)
train_budgets = allocate_budget(TRAIN_TOKEN_BUDGET)
val_budgets = allocate_budget(VAL_TOKEN_BUDGET)

print("Tokenizer words:", word_budgets)
print("Training tokens:", train_budgets)
print("Validation tokens:", val_budgets)

Tokenizer words: {'acts': 500000, 'high_courts': 1000000, 'supreme_court': 1500000, 'tribunals': 1000000}
Training tokens: {'acts': 50000000, 'high_courts': 100000000, 'supreme_court': 150000000, 'tribunals': 100000000}
Validation tokens: {'acts': 62500, 'high_courts': 125000, 'supreme_court': 187500, 'tribunals': 125000}


In [10]:
import os
import json
import hashlib
import sqlite3
import ijson

SPLIT_DIR = os.path.join(OUTPUT_DIR, "document_splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

DEDUP_PATH = os.path.join(SPLIT_DIR, "dedup.sqlite")

# Rebuild from scratch when this cell is rerun.
if os.path.exists(DEDUP_PATH):
    os.remove(DEDUP_PATH)

connection = sqlite3.connect(DEDUP_PATH)
connection.execute(
    "CREATE TABLE seen (digest BLOB PRIMARY KEY) WITHOUT ROWID"
)


class JsonlTexts:
    """Reads texts from disk; each iteration starts from the beginning."""

    def __init__(self, path):
        self.path = path

    def __iter__(self):
        with open(self.path, "r", encoding="utf-8") as f:
            for line in f:
                yield json.loads(line)


train_sources = {}
val_sources = {}
split_counts = {}

try:
    for name, source_path in source_paths.items():
        train_path = os.path.join(SPLIT_DIR, f"{name}_train.jsonl")
        val_path = os.path.join(SPLIT_DIR, f"{name}_val.jsonl")

        counts = {
            "train": 0,
            "validation": 0,
            "invalid": 0,
            "duplicates": 0,
        }

        print(f"\nStreaming {name}...", flush=True)

        with (
            open(source_path, "rb") as source,
            open(train_path, "w", encoding="utf-8") as train_out,
            open(val_path, "w", encoding="utf-8") as val_out,
        ):
            # Verify the top-level structure without loading the file.
            first = source.read(1)
            while first and first.isspace():
                first = source.read(1)

            if first != b"[":
                raise ValueError(f"{name}: expected a top-level JSON list.")

            source.seek(0)

            for index, record in enumerate(
                ijson.items(source, "item"), start=1
            ):
                text = (
                    record.get("text")
                    if isinstance(record, dict)
                    else None
                )

                if not isinstance(text, str) or not text.strip():
                    counts["invalid"] += 1
                    continue

                text = " ".join(text.split())
                digest = hashlib.sha256(text.encode("utf-8")).digest()

                cursor = connection.execute(
                    "INSERT OR IGNORE INTO seen VALUES (?)",
                    (digest,),
                )

                if cursor.rowcount == 0:
                    counts["duplicates"] += 1
                    continue

                # Deterministic document-level split.
                split_digest = hashlib.sha256(
                    str(SEED).encode("ascii") + digest
                ).digest()

                fraction = (
                    int.from_bytes(split_digest[:8], "big") / (1 << 64)
                )

                if fraction < VALIDATION_FRACTION:
                    destination = val_out
                    counts["validation"] += 1
                else:
                    destination = train_out
                    counts["train"] += 1

                destination.write(
                    json.dumps(text, ensure_ascii=False) + "\n"
                )

                if index % 10_000 == 0:
                    connection.commit()

        connection.commit()

        train_sources[name] = JsonlTexts(train_path)
        val_sources[name] = JsonlTexts(val_path)
        split_counts[name] = counts

        print(name, counts, flush=True)

finally:
    connection.close()


Streaming acts...


acts {'train': 17775, 'validation': 956, 'invalid': 4, 'duplicates': 91}



Streaming high_courts...


high_courts {'train': 2297028, 'validation': 121220, 'invalid': 0, 'duplicates': 118490}



Streaming supreme_court...


supreme_court {'train': 52572, 'validation': 2789, 'invalid': 21, 'duplicates': 167}



Streaming tribunals...


tribunals {'train': 820769, 'validation': 43429, 'invalid': 0, 'duplicates': 4934}


In [11]:
def select_word_budget(documents, target_words, source_name):
    selected = []
    collected = 0

    for text in documents:
        remaining = target_words - collected

        if remaining == 0:
            break

        words = text.split()

        if len(words) <= remaining:
            selected.append(text)
            collected += len(words)
        else:
            selected.append(" ".join(words[:remaining]))
            collected += remaining

    if collected != target_words:
        raise ValueError(
            f"{source_name}: needs {target_words:,} words, "
            f"but only {collected:,} are available in its training split."
        )

    return selected

tokenizer_corpus = []

for name, budget in word_budgets.items():
    selected = select_word_budget(
        train_sources[name], budget, name
    )
    tokenizer_corpus.extend(selected)

    print(f"{name}: {budget:,} words")
    del selected

random.Random(SEED).shuffle(tokenizer_corpus)

print("Tokenizer corpus documents:", len(tokenizer_corpus))

acts: 500,000 words
high_courts: 1,000,000 words


supreme_court: 1,500,000 words
tribunals: 1,000,000 words
Tokenizer corpus documents: 2592


In [12]:
BPE_VOCAB_SIZE = 20000

# Always create a fresh instance when retraining.
tokenizer = BytelevelBPE(vocab_size=BPE_VOCAB_SIZE)
tokenizer.train(tokenizer_corpus)


examples = [
    "Section 12(1): The Court dismissed the appeal.",
    "भारत का सर्वोच्च न्यायालय",
    "₹500 — café",
    "Multiple   spaces\nand\ttabs",
]

for text in examples:
    ids = tokenizer.encode(text)


print("Tokenizer prepared.")
print("Actual vocabulary:", tokenizer.get_vocab_size())

del tokenizer_corpus

Building word frequencies...


Unique whitespace-separated words: 122,773


Building initial pair statistics...


Initial unique pairs: 5,425
Target merges: 19,739



Tokenizer training complete.
Final vocabulary size: 20000
Number of merges: 19739
Tokenizer prepared.
Actual vocabulary: 20000


## 6. Create Train and Validation Tokens Set

In [13]:
def write_token_mixture(sources, budgets, output_path):
    total_tokens = sum(budgets.values())

    token_array = np.memmap(
        output_path,
        dtype=np.uint32,
        mode="w+",
        shape=(total_tokens,),
    )

    eos_id = tokenizer.stoi["</s>"]
    vocab_size = tokenizer.get_vocab_size()

    position = 0
    source_ranges = {}

    try:
        for name, target in budgets.items():
            start = position
            written = 0
            documents_used = 0

            print(f"\nEncoding {name}: target {target:,} tokens")

            for text in sources[name]:
                remaining = target - written

                if remaining == 0:
                    break

                ids = tokenizer.encode(text)
                ids.append(eos_id)

                # The final document may be truncated at the token budget.
                chunk = ids[:remaining]

                if not all(0 <= token_id < vocab_size for token_id in chunk):
                    raise ValueError(f"{name}: invalid token ID.")

                count = len(chunk)
                token_array[position:position + count] = np.asarray(
                    chunk, dtype=np.uint32
                )

                position += count
                written += count
                documents_used += 1

            if written != target:
                raise ValueError(
                    f"{name}: needs {target:,} tokens, "
                    f"but only {written:,} are available. "
                    "No documents were repeated."
                )

            source_ranges[name] = {
                "start": start,
                "end": position,
                "tokens": written,
                "documents_used": documents_used,
            }

            print(f"Finished {name}: {written:,} tokens")

        token_array.flush()

    except Exception:
        del token_array
        if os.path.exists(output_path):
            os.remove(output_path)
        raise

    del token_array
    return source_ranges


TRAIN_IDS_PATH = os.path.join(OUTPUT_DIR, "train_ids.uint32")
VAL_IDS_PATH = os.path.join(OUTPUT_DIR, "val_ids.uint32")

train_ranges = write_token_mixture(
    train_sources, train_budgets, TRAIN_IDS_PATH
)

val_ranges = write_token_mixture(
    val_sources, val_budgets, VAL_IDS_PATH
)


Encoding acts: target 50,000,000 tokens


Finished acts: 50,000,000 tokens

Encoding high_courts: target 100,000,000 tokens


Finished high_courts: 100,000,000 tokens

Encoding supreme_court: target 150,000,000 tokens


Finished supreme_court: 150,000,000 tokens

Encoding tribunals: target 100,000,000 tokens


Finished tribunals: 100,000,000 tokens

Encoding acts: target 62,500 tokens


Finished acts: 62,500 tokens

Encoding high_courts: target 125,000 tokens


Finished high_courts: 125,000 tokens

Encoding supreme_court: target 187,500 tokens


Finished supreme_court: 187,500 tokens

Encoding tribunals: target 125,000 tokens


Finished tribunals: 125,000 tokens


In [14]:
train_ids = np.memmap(
    TRAIN_IDS_PATH, dtype=np.uint32, mode="r"
)

val_ids = np.memmap(
    VAL_IDS_PATH, dtype=np.uint32, mode="r"
)


metadata = {
    "seed": SEED,
    "source_ratios": SOURCE_RATIOS,
    "validation_document_fraction": VALIDATION_FRACTION,
    "tokenizer_training_words": TOKENIZER_WORD_BUDGET,
    "vocab_size": tokenizer.get_vocab_size(),
    "dtype": "uint32",
    "train": train_ranges,
    "validation": val_ranges,
}

with open(
    os.path.join(OUTPUT_DIR, "data_metadata.json"),
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, indent=2)

print("Training tokens:", f"{len(train_ids):,}")
print("Validation tokens:", f"{len(val_ids):,}")
print("First IDs:", train_ids[:20])

Training tokens: 400,000,000
Validation tokens: 500,000
First IDs: [ 3672  4332  8794  5985  5875  1973    44  7325   290  9961  5746   625
  6170 10526   536   497   487   273   797  2001]


## 7. Create Token Blocks
A document block is split into a prefix and continuation. The encoder receives only the prefix. The decoder receives BOS plus the continuation shifted right; labels contain the continuation. No second shift is needed in the loss. Padding labels use -100.

Example: prefix = "The court", decoder input = "<s> dismissed the appeal", labels = "dismissed the appeal </s>". EOS is added only at a real document ending.

In [15]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

PAD_ID = tokenizer.stoi["<pad>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]
BLOCK_SIZE = TRAIN_BLOCK_SIZE
MIN_BLOCK = 64


def build_block_index(token_path, index_path, block_size=1024, source_ranges=None):
    ids = np.memmap(token_path, dtype=np.uint32, mode="r")
    ranges = list(source_ranges.values()) if source_ranges else [{"start": 0, "end": len(ids)}]
    block_count = 0
    with open(index_path, "wb") as out:
        for span in ranges:
            span_start, span_end = int(span["start"]), int(span["end"])
            document_start = span_start
            def write_document(end, has_eos):
                nonlocal block_count
                for start in range(document_start, end, block_size):
                    length = min(block_size, end - start)
                    if length >= MIN_BLOCK:
                        real_end = int(has_eos and start + length == end)
                        out.write(np.asarray([start, length, real_end], dtype=np.int64).tobytes())
                        block_count += 1
            for base in range(span_start, span_end, 1_000_000):
                limit = min(base + 1_000_000, span_end)
                for end in np.flatnonzero(ids[base:limit] == EOS_ID) + base:
                    write_document(int(end), True)
                    document_start = int(end) + 1
            if document_start < span_end:
                write_document(span_end, False)
    if block_count == 0:
        raise ValueError("No usable blocks")
    return block_count


class PrefixDataset(Dataset):
    def __init__(self, path, index_path, training):
        self.ids = np.memmap(path, dtype=np.uint32, mode="r")
        self.index = np.memmap(index_path, dtype=np.int64, mode="r").reshape(-1, 3)
        self.training = training

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        start, length, real_end = map(int, self.index[i])
        tokens = torch.tensor(np.asarray(self.ids[start:start + length], dtype=np.int64))
        split = int(torch.randint(length // 4, 3 * length // 4 + 1, ())) if self.training else length // 2
        src = torch.cat([torch.tensor([BOS_ID]), tokens[:split]])
        suffix = tokens[split:]
        if real_end:
            suffix = torch.cat([suffix, torch.tensor([EOS_ID])])
        decoder_input = torch.cat([torch.tensor([BOS_ID]), suffix[:-1]])
        return src, decoder_input, suffix


def collate_prefix(items):
    src, dec, labels = zip(*items)
    return {
        "src_ids": pad_sequence(src, batch_first=True, padding_value=PAD_ID),
        "decoder_ids": pad_sequence(dec, batch_first=True, padding_value=PAD_ID),
        "labels": pad_sequence(labels, batch_first=True, padding_value=-100),
    }

## 8. Create Datasets

In [16]:
train_index_path = os.path.join(OUTPUT_DIR, "train_blocks.int64")
val_index_path = os.path.join(OUTPUT_DIR, "val_blocks.int64")
print("Train blocks:", build_block_index(TRAIN_IDS_PATH, train_index_path, block_size=BLOCK_SIZE, source_ranges=train_ranges))
print("Validation blocks:", build_block_index(VAL_IDS_PATH, val_index_path, block_size=BLOCK_SIZE, source_ranges=val_ranges))
train_dataset = PrefixDataset(TRAIN_IDS_PATH, train_index_path, True)
val_dataset = PrefixDataset(VAL_IDS_PATH, val_index_path, False)

Train blocks: 559416
Validation blocks: 663


## 9. Define The Model

In [17]:
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class TransformerConfig:
    vocab_size: int
    pad_id: int
    d_model: int = 512
    n_heads: int = 8
    d_ff: int = 2048
    n_layers: int = 6
    dropout: float = 0.1
    max_src_len: int = 2048
    max_tgt_len: int = 2048
    pos_emb_type: str = "rope"
    rope_base: float = 10000.0


class EmbeddingDropout(nn.Module):
    # Preserve the embedding dropout and configured length limit without adding PE.
    def __init__(self, max_len, dropout):
        super().__init__()
        self.max_len = max_len
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        if x.size(1) > self.max_len:
            raise ValueError('Sequence exceeds configured context length')
        return self.dropout(x)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        if head_dim % 2:
            raise ValueError('RoPE requires an even head dimension')
        inv_freq = base ** (
            -torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim
        )
        self.register_buffer('inv_freq', inv_freq, persistent=False)

    def forward(self, x):
        # x: [batch, heads, length, head_dim]. Split-half pairs: (i, i + d/2).
        # Generation recomputes the full decoder sequence, so positions start at 0.
        with torch.autocast(device_type=x.device.type, enabled=False):
            positions = torch.arange(x.size(2), device=x.device, dtype=torch.float32)
            angles = torch.outer(positions, self.inv_freq.float())
            cos = angles.cos()[None, None, :, :]
            sin = angles.sin()[None, None, :, :]
        cos, sin = cos.to(x.dtype), sin.to(x.dtype)
        first, second = x.chunk(2, dim=-1)
        return torch.cat((first * cos - second * sin,
                          first * sin + second * cos), dim=-1)


In [18]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout, use_rope=False, rope_base=10000.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.rope = RotaryEmbedding(self.d_head, rope_base) if use_rope else None
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        # [batch, length, d_model] -> [batch, heads, length, d_head]
        b, length, _ = x.shape
        return x.reshape(b, length, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, query, key, value, allowed, return_attention=False):
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))
        if self.rope is not None:
            q = self.rope(q)
            k = self.rope(k)
        # True means this key may be attended to, including the causal mask.
        weights = None
        if return_attention:
            # Keep explicit probabilities for attention visualizations.
            with torch.autocast(device_type=q.device.type, enabled=False):
                scores = (q.float() @ k.float().transpose(-2, -1)) / math.sqrt(self.d_head)
                scores = scores.masked_fill(~allowed, float('-inf'))
                weights = scores.softmax(dim=-1)
            context = self.dropout(weights.to(v.dtype)) @ v
        else:
            # Let PyTorch select a supported backend; dropout must be 0 at eval.
            context = F.scaled_dot_product_attention(
                q, k, v, attn_mask=allowed,
                dropout_p=self.dropout.p if self.training else 0.0,
                is_causal=False,  # Causality is already present in allowed.
            )
        b, _, length, _ = context.shape
        context = context.transpose(1, 2).contiguous().reshape(b, length, -1)
        output = self.out_proj(context)
        return output, weights.mean(dim=1) if return_attention else None

In [19]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=True, rope_base=cfg.rope_base)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, source_mask):
        attended, _ = self.attention(x, x, x, source_mask)
        x = self.norm1(x + self.dropout(attended))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=True, rope_base=cfg.rope_base)
        self.cross_attention = MultiHeadAttention(
            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=False)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.norm3 = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, memory, target_mask, source_mask, return_attention=False):
        attended, _ = self.self_attention(x, x, x, target_mask)
        x = self.norm1(x + self.dropout(attended))
        attended, weights = self.cross_attention(x, memory, memory, source_mask,
                                                  return_attention)
        x = self.norm2(x + self.dropout(attended))
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x, weights

In [20]:
class LegalTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        if cfg.pos_emb_type != 'rope':
            raise ValueError('This architecture expects a RoPE configuration')
        self.cfg = cfg
        # Separate weights make the three roles easy to inspect.
        self.source_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.target_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)
        self.source_position = EmbeddingDropout(cfg.max_src_len, cfg.dropout)
        self.target_position = EmbeddingDropout(cfg.max_tgt_len, cfg.dropout)
        self.encoder = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.n_layers)])
        self.output = nn.Linear(cfg.d_model, cfg.vocab_size)
        for parameter in self.parameters():
            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)
        with torch.no_grad():
            self.source_embedding.weight[cfg.pad_id].zero_()
            self.target_embedding.weight[cfg.pad_id].zero_()

    def encode(self, source_ids):
        source_mask = source_ids.ne(self.cfg.pad_id)[:, None, None, :]
        x = self.source_embedding(source_ids) * math.sqrt(self.cfg.d_model)
        x = self.source_position(x)
        for layer in self.encoder:
            x = layer(x, source_mask)
        return x, source_mask

    def decode(self, target_ids, memory, source_mask, return_attention=False):
        length = target_ids.size(1)
        causal = torch.ones(length, length, dtype=torch.bool, device=target_ids.device).tril()
        valid_keys = target_ids.ne(self.cfg.pad_id)[:, None, None, :]
        target_mask = valid_keys & causal[None, None, :, :]
        x = self.target_embedding(target_ids) * math.sqrt(self.cfg.d_model)
        x = self.target_position(x)
        weights = None
        for index, layer in enumerate(self.decoder):
            x, weights = layer(x, memory, target_mask, source_mask,
                               return_attention and index == len(self.decoder) - 1)
        logits = self.output(x)
        return (logits, weights) if return_attention else logits

    def forward(self, source_ids, target_ids, return_attention=False):
        memory, source_mask = self.encode(source_ids)
        return self.decode(target_ids, memory, source_mask, return_attention)

## 10. Training Arguments
Batch size and gradient accumulation are set in Configuration. For the default batch 2 and accumulation 4, an update combines up to 8 examples. A two-GPU batch is divided across GPUs, not multiplied by their count.

In [21]:
def unwrap_model(model):
    return model.module if isinstance(model, torch.nn.DataParallel) else model

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = TransformerConfig(vocab_size=tokenizer.get_vocab_size(), pad_id=PAD_ID, max_src_len=TRAIN_BLOCK_SIZE, max_tgt_len=TRAIN_BLOCK_SIZE)
model = LegalTransformer(cfg).to(device)
base_model = unwrap_model(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

# Conservative starting microbatch; increase after measuring GPU memory.
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=collate_prefix, pin_memory=device.type == "cuda")
val_loader = DataLoader(val_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=False,
                        num_workers=0, collate_fn=collate_prefix, pin_memory=device.type == "cuda")
batch = next(iter(train_loader))
for name, value in batch.items():
    print(name, value.shape)
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
print(f"Planned microbatches: {TRAIN_STEPS:,}")
print(f"Planned optimizer updates: {math.ceil(TRAIN_STEPS / GRAD_ACCUM_STEPS):,}")
print(f"Approximate data passes: {TRAIN_STEPS / len(train_loader):.2f}")


Parameters: 74,841,632
src_ids torch.Size([4, 533])
decoder_ids torch.Size([4, 504])
labels torch.Size([4, 504])
Planned microbatches: 80,000
Planned optimizer updates: 5,000
Approximate data passes: 0.57


In [22]:
# Small CPU check before committing GPU time; preserves the training RNG.
RUN_MODEL_CHECK = True
if RUN_MODEL_CHECK:
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(123)
        attention = MultiHeadAttention(16, 4, dropout=0.2, use_rope=True).eval()
        cross_attention = MultiHeadAttention(16, 4, dropout=0.2, use_rope=False).eval()
        query = torch.randn(2, 4, 16)
        memory = torch.randn(2, 5, 16)
        self_keys = torch.tensor([[True, True, True, False],
                                  [True, True, True, True]])[:, None, None, :]
        cross_keys = torch.tensor([[True, True, True, False, False],
                                   [True, True, True, True, True]])[:, None, None, :]
        causal = torch.ones(4, 4, dtype=torch.bool).tril()[None, None]
        with torch.no_grad():
            for keys, mask in [(query, self_keys),
                               (query, self_keys & causal),
                               (memory, cross_keys)]:
                module = attention if keys is query else cross_attention
                fast, _ = module(query, keys, keys, mask)
                explicit, weights = module(query, keys, keys, mask, True)
                torch.testing.assert_close(fast, explicit, rtol=1e-4, atol=1e-5)
                assert torch.isfinite(weights).all()

        tiny_cfg = TransformerConfig(vocab_size=32, pad_id=0, d_model=16,
            n_heads=4, d_ff=32, n_layers=2, dropout=0,
            max_src_len=16, max_tgt_len=16)
        tiny = LegalTransformer(tiny_cfg).eval()
        assert all(layer.attention.rope is not None for layer in tiny.encoder)
        assert all(layer.self_attention.rope is not None for layer in tiny.decoder)
        assert all(layer.cross_attention.rope is None for layer in tiny.decoder)
        with torch.no_grad():
            rotary = RotaryEmbedding(4)
            probe = torch.randn(1, 2, 2048, 4)
            rotated = rotary(probe)
            torch.testing.assert_close(rotated[:, :, 0], probe[:, :, 0])
            torch.testing.assert_close(rotated.square().sum(-1),
                                       probe.square().sum(-1), rtol=1e-5, atol=1e-5)
            assert torch.isfinite(rotated).all()
        del rotary, probe, rotated
        src = torch.tensor([[2, 4, 5, 0], [2, 8, 9, 10]])
        dec = torch.tensor([[2, 6, 7], [2, 11, 0]])
        with torch.no_grad():
            logits = tiny(src, dec)
            changed = dec.clone()
            changed[0, -1] = 8
            torch.testing.assert_close(logits[:, :-1], tiny(src, changed)[:, :-1])
            plain, _ = tiny(src, dec, return_attention=True)
            torch.testing.assert_close(logits, plain, rtol=1e-4, atol=1e-5)
        tiny.train()
        labels = torch.tensor([[6, 7, 3], [11, 3, -100]])
        check_loss = F.cross_entropy(tiny(src, dec).flatten(0, 1),
                                     labels.flatten(), ignore_index=-100)
        check_loss.backward()
        assert torch.isfinite(check_loss)
        assert all(p.grad is not None and torch.isfinite(p.grad).all()
                   for p in tiny.parameters() if p.requires_grad)
        print('Passed: RoPE at 2048 positions, attention placement, SDPA agreement, causality and backward.')
        del tiny, attention, cross_attention, module, query, memory, logits, plain, check_loss, weights


Passed: RoPE at 2048 positions, attention placement, SDPA agreement, causality and backward.


## 11. Training
Use the paper-style warmup/inverse-square-root schedule, Adam (0.9, 0.98),
token-normalized accumulation and AMP. Label smoothing defaults to **0.0**;
validation always measures unsmoothed NLL and conditional suffix perplexity.
Training and validation still differ in dropout and prefix sampling.
Validation occurs only at optimizer-update boundaries; best validation weights
are restored at the end, even if the last update was worse.

`TRAIN_STEPS` counts microbatches. `RESUME_FROM = None` starts fresh. To resume,
use this notebook's `latest.pt` and keep its companion `best.pt` in the same folder.
Resume restores optimizer/scaler, schedule position and logged history, then
starts a new shuffled data pass. Checkpoints contain tokenizer/data-setting
signatures to reject incompatible resumes; they do not fingerprint every dataset
byte. Use exactly the same source data. Old notebook checkpoints are not supported
by this resume route. Preserve the checkpoint folder before ending a Kaggle session.


In [ ]:
import hashlib

AMP_ENABLED = device.type == "cuda"
AMP_DTYPE = torch.bfloat16 if AMP_ENABLED and torch.cuda.is_bf16_supported() else torch.float16


def save_atomic(state, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    torch.save(state, path + ".tmp")
    os.replace(path + ".tmp", path)


@torch.inference_mode()
def evaluate_validation(model, val_loader, device):
    was_training = model.training
    model.eval()
    nll, tokens = 0.0, 0
    try:
        for batch in val_loader:
            src, dec, labels = [batch[k].to(device) for k in ("src_ids", "decoder_ids", "labels")]
            with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
                logits = model(src, dec)
                loss = F.cross_entropy(logits.flatten(0, 1), labels.flatten(), ignore_index=-100, reduction="sum")
            nll += loss.item()
            tokens += labels.ne(-100).sum().item()
    finally:
        model.train(was_training)
    if tokens == 0:
        raise ValueError("Empty validation set")
    loss = nll / tokens
    return loss, math.exp(loss) if loss < 700 else float("inf")


def train_model(model, train_loader, val_loader, train_steps=100000, lr_factor=0.15,
                accumulation_steps=16, log_every=100, eval_every=1000,
                resume_from=None):
    if train_steps < 1 or accumulation_steps < 1 or len(train_loader) == 0:
        raise ValueError("Positive training steps, accumulation, and nonempty loader required")
    os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED and AMP_DTYPE == torch.float16)
    def learning_rate(update):
        step = update + 1
        return lr_factor * unwrap_model(model).cfg.d_model ** -0.5 * min(
            step ** -0.5, step * WARMUP_UPDATES ** -1.5)
    # Token identities matter even when two tokenizers have the same vocab size.
    tokenizer_signature = hashlib.sha256(json.dumps(
        dict(vocab=tokenizer.vocab, merges=tokenizer.merges, specials=tokenizer.specials),
        ensure_ascii=False, sort_keys=True,
    ).encode("utf-8")).hexdigest()
    data_signature = hashlib.sha256(json.dumps(
        dict(metadata=metadata, block_size=BLOCK_SIZE, min_block=MIN_BLOCK,
             train_blocks=len(train_loader.dataset), val_blocks=len(val_loader.dataset)),
        sort_keys=True,
    ).encode("utf-8")).hexdigest()
    run_settings = dict(batch_size=train_loader.batch_size,
                        accumulation_steps=accumulation_steps, lr_factor=lr_factor,
                        warmup_updates=WARMUP_UPDATES, label_smoothing=LABEL_SMOOTHING,
                        seed=SEED, amp_dtype=str(AMP_DTYPE), amp_enabled=AMP_ENABLED)
    best = float("inf")
    optimizer_steps = 0
    loss_history, step_history, validation_history = [], [], []
    log_nll = log_tokens = pending_tokens = pending_batches = 0
    evaluation_due = False
    start_step = 0
    
    iterator = iter(train_loader)
    # Scale losses by this constant during backward, then normalize exactly.
    reference_tokens = train_loader.batch_size * BLOCK_SIZE * accumulation_steps
    optimizer.zero_grad(set_to_none=True)
    model.train()
    for step in range(start_step + 1, train_steps + 1):
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(train_loader)
            batch = next(iterator)
        src, dec, labels = [batch[k].to(device) for k in ("src_ids", "decoder_ids", "labels")]
        count = int(labels.ne(-100).sum())
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=AMP_ENABLED):
            logits = model(src, dec)
            loss = F.cross_entropy(logits.flatten(0, 1), labels.flatten(), ignore_index=-100, reduction="sum",
                                   label_smoothing=LABEL_SMOOTHING)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Nonfinite loss at batch {step}")
        scaler.scale(loss / reference_tokens).backward()
        log_nll += loss.item()
        log_tokens += count
        pending_tokens += count
        pending_batches += 1
        del logits, loss, src, dec, labels
        if step == 1 or step % log_every == 0 or step == train_steps:
            step_history.append(step)
            loss_history.append(log_nll / log_tokens)
            print(f"Batch {step}/{train_steps} | training loss {loss_history[-1]:.4f}", flush=True)
            log_nll = log_tokens = 0
        evaluation_due |= step % eval_every == 0
        if pending_batches == accumulation_steps or step == train_steps:
            scaler.unscale_(optimizer)
            for parameter in model.parameters():
                if parameter.grad is not None:
                    parameter.grad.mul_(reference_tokens / pending_tokens)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            for group in optimizer.param_groups:
                group["lr"] = learning_rate(optimizer_steps)
            previous_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= previous_scale:
                optimizer_steps += 1
            optimizer.zero_grad(set_to_none=True)
            pending_tokens = pending_batches = 0
            if evaluation_due or step == train_steps:
                val_loss, val_ppl = evaluate_validation(model, val_loader, device)
                validation_history.append(dict(step=step, optimizer_steps=optimizer_steps, val_loss=val_loss, val_ppl=val_ppl))
                state = dict(model_config=asdict(unwrap_model(model).cfg), model_state_dict=unwrap_model(model).state_dict(),
                             step=step, val_loss=val_loss)
                if val_loss < best:
                    best = val_loss
                    save_atomic(state, os.path.join(MODEL_OUTPUT_DIR, "best.pt"))
                state.update(optimizer_state_dict=optimizer.state_dict(), scaler_state_dict=scaler.state_dict(),
                             optimizer_steps=optimizer_steps, best_val_loss=best,
                             train_steps=train_steps, accumulation_steps=accumulation_steps,
                             seed=SEED, lr_factor=lr_factor, warmup_updates=WARMUP_UPDATES,
                             label_smoothing=LABEL_SMOOTHING, torch_rng_state=torch.get_rng_state(),
                             cuda_rng_state=torch.cuda.get_rng_state_all() if AMP_ENABLED else [])
                state.update(
                    resume_format=1, tokenizer_signature=tokenizer_signature,
                    data_signature=data_signature, run_settings=run_settings,
                    python_rng_state=random.getstate(), log_nll=log_nll, log_tokens=log_tokens,
                    history=dict(train_steps=step_history, train_losses=loss_history,
                                 validation=validation_history),
                )
                save_atomic(state, os.path.join(MODEL_OUTPUT_DIR, "latest.pt"))
                with open(os.path.join(MODEL_OUTPUT_DIR, "history.json"), "w") as f:
                    json.dump(dict(train_steps=step_history, train_losses=loss_history,
                                   validation=validation_history), f, indent=2)
                print(f"Updates {optimizer_steps} | validation loss {val_loss:.4f} | conditional PPL {val_ppl:.2f}", flush=True)
                evaluation_due = False
    state = torch.load(os.path.join(MODEL_OUTPUT_DIR, "best.pt"), map_location="cpu", weights_only=True)
    unwrap_model(model).load_state_dict(state["model_state_dict"])
    model.eval()
    return loss_history, step_history, validation_history

In [ ]:
loss_history, step_history, validation_history = train_model(
    model, train_loader, val_loader, train_steps=TRAIN_STEPS,
    lr_factor=LR_FACTOR, accumulation_steps=GRAD_ACCUM_STEPS,
    log_every=100, eval_every=1000, resume_from=RESUME_FROM,
)

## 12. Perplexity

In [ ]:
val_loss, val_perplexity = evaluate_validation(model, val_loader, device)
print(f"Conditional suffix loss: {val_loss:.4f}; perplexity: {val_perplexity:.2f}")

## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

val_steps = [record["step"] for record in validation_history]
val_losses = [record["val_loss"] for record in validation_history]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    step_history,
    loss_history,
    marker=".",
    label=f"Training loss (smoothing={LABEL_SMOOTHING:g})",
)

ax.plot(
    val_steps,
    val_losses,
    marker="o",
    linestyle="--",
    label="Validation NLL (no smoothing)",
)

ax.set_xlabel("Training step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Prefix-Conditioned Training and Validation Loss")
ax.grid(alpha=0.3)
ax.legend()

fig.tight_layout()
fig.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

print("Plot saved:", LOSS_PLOT_PATH)

## 14. Save Final Model to Kaggle Output

In [ ]:
import os
import json
import shutil
import torch
from dataclasses import asdict

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

# 1. Save model configuration and weights.
# Your training function restored the best evaluated weights.
model_path = os.path.join(MODEL_OUTPUT_DIR, "model.pt")

torch.save(
    {
        "model_config": asdict(unwrap_model(model).cfg),
        "model_state_dict": {
            name: tensor.detach().cpu()
            for name, tensor in unwrap_model(model).state_dict().items()
        },
    },
    model_path,
)

# 2. Save tokenizer vocabulary, merges, and byte representations.
tokenizer_path = os.path.join(MODEL_OUTPUT_DIR, "tokenizer.json")

tokenizer_state = {
    "vocab_size": tokenizer.get_vocab_size(),
    "specials": tokenizer.specials,
    "vocab": tokenizer.vocab,
    "merges": [list(pair) for pair in tokenizer.merges],
    "merge_ranks": [
        {
            "pair": list(pair),
            "rank": rank,
            "merged_symbol": merged_symbol,
        }
        for pair, (rank, merged_symbol)
        in tokenizer.merge_ranks.items()
    ],
    "symbol_to_bytes": {
        symbol: list(byte_value)
        for symbol, byte_value in tokenizer.symbol_to_bytes.items()
    },
}

with open(tokenizer_path, "w", encoding="utf-8") as f:
    json.dump(tokenizer_state, f, ensure_ascii=False)


print("Model saved:", model_path)
print("Tokenizer saved:", tokenizer_path)
print("Model and tokenizer saved.")
# Keep settings and history next to model.pt for later comparison and ZIP export.
training_settings = dict(
    train_microbatches=TRAIN_STEPS, batch_size=TRAIN_BATCH_SIZE,
    accumulation_steps=GRAD_ACCUM_STEPS, warmup_updates=WARMUP_UPDATES,
    lr_factor=LR_FACTOR, label_smoothing=LABEL_SMOOTHING,
    train_token_budget=TRAIN_TOKEN_BUDGET, val_token_budget=VAL_TOKEN_BUDGET,
    seed=SEED, attention="sdpa", resume_from=RESUME_FROM,
    block_size=TRAIN_BLOCK_SIZE,
    pos_emb_type=unwrap_model(model).cfg.pos_emb_type,
    rope_base=unwrap_model(model).cfg.rope_base,
)
with open(os.path.join(MODEL_OUTPUT_DIR, "training_config.json"), "w") as f:
    json.dump(training_settings, f, indent=2)
with open(os.path.join(MODEL_OUTPUT_DIR, "history.json"), "w") as f:
    json.dump(dict(train_steps=step_history, train_losses=loss_history,
                   validation=validation_history), f, indent=2)
print("History and training settings saved in:", MODEL_OUTPUT_DIR)


In [ ]:
architecture_source = 'import math\nfrom dataclasses import dataclass, asdict\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n@dataclass\nclass TransformerConfig:\n    vocab_size: int\n    pad_id: int\n    d_model: int = 512\n    n_heads: int = 8\n    d_ff: int = 2048\n    n_layers: int = 6\n    dropout: float = 0.1\n    max_src_len: int = 2048\n    max_tgt_len: int = 2048\n    pos_emb_type: str = "rope"\n    rope_base: float = 10000.0\n\n\nclass EmbeddingDropout(nn.Module):\n    # Preserve the embedding dropout and configured length limit without adding PE.\n    def __init__(self, max_len, dropout):\n        super().__init__()\n        self.max_len = max_len\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x):\n        if x.size(1) > self.max_len:\n            raise ValueError(\'Sequence exceeds configured context length\')\n        return self.dropout(x)\n\n\nclass RotaryEmbedding(nn.Module):\n    def __init__(self, head_dim, base=10000.0):\n        super().__init__()\n        if head_dim % 2:\n            raise ValueError(\'RoPE requires an even head dimension\')\n        inv_freq = base ** (\n            -torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim\n        )\n        self.register_buffer(\'inv_freq\', inv_freq, persistent=False)\n\n    def forward(self, x):\n        # x: [batch, heads, length, head_dim]. Split-half pairs: (i, i + d/2).\n        # Generation recomputes the full decoder sequence, so positions start at 0.\n        with torch.autocast(device_type=x.device.type, enabled=False):\n            positions = torch.arange(x.size(2), device=x.device, dtype=torch.float32)\n            angles = torch.outer(positions, self.inv_freq.float())\n            cos = angles.cos()[None, None, :, :]\n            sin = angles.sin()[None, None, :, :]\n        cos, sin = cos.to(x.dtype), sin.to(x.dtype)\n        first, second = x.chunk(2, dim=-1)\n        return torch.cat((first * cos - second * sin,\n                          first * sin + second * cos), dim=-1)\n\nclass MultiHeadAttention(nn.Module):\n    def __init__(self, d_model, n_heads, dropout, use_rope=False, rope_base=10000.0):\n        super().__init__()\n        self.n_heads = n_heads\n        self.d_head = d_model // n_heads\n        self.rope = RotaryEmbedding(self.d_head, rope_base) if use_rope else None\n        self.q_proj = nn.Linear(d_model, d_model, bias=False)\n        self.k_proj = nn.Linear(d_model, d_model, bias=False)\n        self.v_proj = nn.Linear(d_model, d_model, bias=False)\n        self.out_proj = nn.Linear(d_model, d_model, bias=False)\n        self.dropout = nn.Dropout(dropout)\n\n    def split_heads(self, x):\n        # [batch, length, d_model] -> [batch, heads, length, d_head]\n        b, length, _ = x.shape\n        return x.reshape(b, length, self.n_heads, self.d_head).transpose(1, 2)\n\n    def forward(self, query, key, value, allowed, return_attention=False):\n        q = self.split_heads(self.q_proj(query))\n        k = self.split_heads(self.k_proj(key))\n        v = self.split_heads(self.v_proj(value))\n        if self.rope is not None:\n            q = self.rope(q)\n            k = self.rope(k)\n        # True means this key may be attended to, including the causal mask.\n        weights = None\n        if return_attention:\n            # Keep explicit probabilities for attention visualizations.\n            with torch.autocast(device_type=q.device.type, enabled=False):\n                scores = (q.float() @ k.float().transpose(-2, -1)) / math.sqrt(self.d_head)\n                scores = scores.masked_fill(~allowed, float(\'-inf\'))\n                weights = scores.softmax(dim=-1)\n            context = self.dropout(weights.to(v.dtype)) @ v\n        else:\n            # Let PyTorch select a supported backend; dropout must be 0 at eval.\n            context = F.scaled_dot_product_attention(\n                q, k, v, attn_mask=allowed,\n                dropout_p=self.dropout.p if self.training else 0.0,\n                is_causal=False,  # Causality is already present in allowed.\n            )\n        b, _, length, _ = context.shape\n        context = context.transpose(1, 2).contiguous().reshape(b, length, -1)\n        output = self.out_proj(context)\n        return output, weights.mean(dim=1) if return_attention else None\n\nclass FeedForward(nn.Module):\n    def __init__(self, d_model, d_ff):\n        super().__init__()\n        self.linear1 = nn.Linear(d_model, d_ff)\n        self.linear2 = nn.Linear(d_ff, d_model)\n\n    def forward(self, x):\n        return self.linear2(F.relu(self.linear1(x)))\n\nclass EncoderLayer(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.attention = MultiHeadAttention(\n            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=True, rope_base=cfg.rope_base)\n        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)\n        self.norm1 = nn.LayerNorm(cfg.d_model)\n        self.norm2 = nn.LayerNorm(cfg.d_model)\n        self.dropout = nn.Dropout(cfg.dropout)\n\n    def forward(self, x, source_mask):\n        attended, _ = self.attention(x, x, x, source_mask)\n        x = self.norm1(x + self.dropout(attended))\n        x = self.norm2(x + self.dropout(self.ffn(x)))\n        return x\n\nclass DecoderLayer(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.self_attention = MultiHeadAttention(\n            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=True, rope_base=cfg.rope_base)\n        self.cross_attention = MultiHeadAttention(\n            cfg.d_model, cfg.n_heads, cfg.dropout, use_rope=False)\n        self.ffn = FeedForward(cfg.d_model, cfg.d_ff)\n        self.norm1 = nn.LayerNorm(cfg.d_model)\n        self.norm2 = nn.LayerNorm(cfg.d_model)\n        self.norm3 = nn.LayerNorm(cfg.d_model)\n        self.dropout = nn.Dropout(cfg.dropout)\n\n    def forward(self, x, memory, target_mask, source_mask, return_attention=False):\n        attended, _ = self.self_attention(x, x, x, target_mask)\n        x = self.norm1(x + self.dropout(attended))\n        attended, weights = self.cross_attention(x, memory, memory, source_mask,\n                                                  return_attention)\n        x = self.norm2(x + self.dropout(attended))\n        x = self.norm3(x + self.dropout(self.ffn(x)))\n        return x, weights\n\nclass LegalTransformer(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        if cfg.pos_emb_type != \'rope\':\n            raise ValueError(\'This architecture expects a RoPE configuration\')\n        self.cfg = cfg\n        # Separate weights make the three roles easy to inspect.\n        self.source_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)\n        self.target_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)\n        self.source_position = EmbeddingDropout(cfg.max_src_len, cfg.dropout)\n        self.target_position = EmbeddingDropout(cfg.max_tgt_len, cfg.dropout)\n        self.encoder = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.n_layers)])\n        self.decoder = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.n_layers)])\n        self.output = nn.Linear(cfg.d_model, cfg.vocab_size)\n        for parameter in self.parameters():\n            if parameter.dim() > 1:\n                nn.init.xavier_uniform_(parameter)\n        with torch.no_grad():\n            self.source_embedding.weight[cfg.pad_id].zero_()\n            self.target_embedding.weight[cfg.pad_id].zero_()\n\n    def encode(self, source_ids):\n        source_mask = source_ids.ne(self.cfg.pad_id)[:, None, None, :]\n        x = self.source_embedding(source_ids) * math.sqrt(self.cfg.d_model)\n        x = self.source_position(x)\n        for layer in self.encoder:\n            x = layer(x, source_mask)\n        return x, source_mask\n\n    def decode(self, target_ids, memory, source_mask, return_attention=False):\n        length = target_ids.size(1)\n        causal = torch.ones(length, length, dtype=torch.bool, device=target_ids.device).tril()\n        valid_keys = target_ids.ne(self.cfg.pad_id)[:, None, None, :]\n        target_mask = valid_keys & causal[None, None, :, :]\n        x = self.target_embedding(target_ids) * math.sqrt(self.cfg.d_model)\n        x = self.target_position(x)\n        weights = None\n        for index, layer in enumerate(self.decoder):\n            x, weights = layer(x, memory, target_mask, source_mask,\n                               return_attention and index == len(self.decoder) - 1)\n        logits = self.output(x)\n        return (logits, weights) if return_attention else logits\n\n    def forward(self, source_ids, target_ids, return_attention=False):\n        memory, source_mask = self.encode(source_ids)\n        return self.decode(target_ids, memory, source_mask, return_attention)\n'
with open(os.path.join(MODEL_OUTPUT_DIR, "encoder_decoder.py"), "w", encoding="utf-8") as f:
    f.write(architecture_source)


In [ ]:
import shutil
from IPython.display import FileLink, display

zip_path = shutil.make_archive(
    "/kaggle/working/model_output",
    "zip",
    root_dir=MODEL_OUTPUT_DIR,
)

display(FileLink(zip_path))

## 15. Push Model to Hugging Face Hub

In [ ]:
# Optional: upload your saved model folder yourself after reviewing it.
# from huggingface_hub import HfApi
# HfApi().upload_folder(
#     folder_path=MODEL_OUTPUT_DIR,
#     repo_id="your-username/your-encoder-decoder-model",
#     repo_type="model",
# )

## 16. Load Test Data from HuggingFace

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)



print("Test examples:", len(test_data))

## 18. Load Final Model for Generation

In [ ]:
# Release the training model before loading another full copy onto the GPU.
model.to('cpu')
if device.type == 'cuda':
    torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(
    os.path.join(MODEL_OUTPUT_DIR, "model.pt"),
    map_location="cpu",
    weights_only=True,
)

generation_cfg = TransformerConfig(**checkpoint["model_config"])
generation_model = LegalTransformer(generation_cfg)

generation_model.load_state_dict(checkpoint["model_state_dict"])
del checkpoint

generation_model = generation_model.to(device).eval()

with open(
    os.path.join(MODEL_OUTPUT_DIR, "tokenizer.json"),
    "r",
    encoding="utf-8",
) as f:
    state = json.load(f)

generation_tokenizer = BytelevelBPE(vocab_size=state["vocab_size"])
generation_tokenizer.specials = state["specials"]
generation_tokenizer.vocab = state["vocab"]
generation_tokenizer.vocab_size = len(state["vocab"])

generation_tokenizer.stoi = {
    symbol: i for i, symbol in enumerate(state["vocab"])
}
generation_tokenizer.itos = {
    i: symbol for symbol, i in generation_tokenizer.stoi.items()
}

generation_tokenizer.merges = [
    tuple(pair) for pair in state["merges"]
]
generation_tokenizer.merge_ranks = {
    tuple(record["pair"]): (
        record["rank"],
        record["merged_symbol"],
    )
    for record in state["merge_ranks"]
}
generation_tokenizer.symbol_to_bytes = {
    symbol: bytes(values)
    for symbol, values in state["symbol_to_bytes"].items()
}


print("Model and tokenizer loaded.")

## 19. Generate continuations
Same decoding modes and word counting as your reference notebook. The encoder runs once; the decoder runs on the growing continuation. Special tokens are suppressed for fixed 128-word output. The decoder's configured context length is enforced here, so the actual output token budget is the smaller of max_new_tokens and max_tgt_len. Incomplete output raises a diagnostic error instead of looping or inventing filler.

In [ ]:
@torch.no_grad()
def generate_continuation(
    prompt, model, tokenizer, device,
    target_words=128, mode='greedy', temperature=0.8, top_k=50,
    max_new_tokens=1024,
):
    model.eval()
    if mode not in ('greedy','temperature','top_k') or temperature <= 0 or top_k < 1:
        raise ValueError('Check decoding mode, temperature and top_k.')
    ids = tokenizer.encode(prompt)
    if not ids:
        raise ValueError('Input produced zero BPE tokens.')
    bos = tokenizer.stoi['<s>']
    input_ids = torch.tensor([[bos]+ids[-(model.cfg.max_src_len-1):]],dtype=torch.long,device=device)
    decoder_ids = torch.tensor([[bos]],dtype=torch.long,device=device)
    generated = []
    word_count = word_bytes = 0
    # Cache tokenizer metadata once instead of scanning merges at every step.
    if not hasattr(tokenizer,'generation_tables'):
        symbols = [tokenizer.itos[i] for i in range(tokenizer.get_vocab_size())]
        tokenizer.generation_tables = (
            [tokenizer._symbol_ends_word(s) for s in symbols],
            [len(tokenizer.symbol_to_bytes.get(s,b'')) for s in symbols],
            [tokenizer.stoi[s] for s in tokenizer.specials],
        )
    ends_word,byte_lengths,special_ids = tokenizer.generation_tables
    empty_ids = [i for i,n in enumerate(byte_lengths) if n == 0]
    use_amp = device.type == 'cuda'
    with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=use_amp):
        memory,source_mask = model.encode(input_ids)
    token_limit = min(max_new_tokens,model.cfg.max_tgt_len)
    for step in range(token_limit):
        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=use_amp):
            logits = model.decode(decoder_ids,memory,source_mask)
        next_logits = logits[:,-1,:].float()
        if generated:
            recent_ids = torch.tensor(
                list(set(generated[-128:])),
                dtype=torch.long,
                device=next_logits.device,
            )
        
            values = next_logits[:, recent_ids]
            penalty = 1.15
        
            next_logits[:, recent_ids] = torch.where(
                values < 0,
                values * penalty,
                values / penalty,
            )
        next_logits[:,special_ids] = float('-inf')
        if word_bytes == 0:
            next_logits[:,empty_ids] = float('-inf')
        if mode == 'greedy':
            next_token = next_logits.argmax(dim=-1,keepdim=True)
        elif mode == 'temperature':
            probs = (next_logits/temperature).softmax(dim=-1)
            next_token = torch.multinomial(probs,1)
        else:
            values,indices = torch.topk(next_logits/temperature,
                min(top_k,next_logits.size(-1)),dim=-1)
            choice = torch.multinomial(values.softmax(dim=-1),1)
            next_token = indices.gather(-1,choice)
        token_id = next_token.item()
        generated.append(token_id)
        word_bytes += byte_lengths[token_id]
        if ends_word[token_id]:
            if word_bytes > 0: word_count += 1
            word_bytes = 0
            if word_count >= target_words:
                words = tokenizer.decode(generated).split()
                if len(words) >= target_words:
                    return ' '.join(words[:target_words])
        decoder_ids = torch.cat([decoder_ids,next_token],dim=1)
    text = tokenizer.decode(generated)
    raise RuntimeError(f'Only {len(text.split())} decoded words after {len(generated)} tokens. Preview: {text[:500]!r}')

## 20. Generate Predictions for the Test Set

In [ ]:
GENERATION_MODE = 'top_k'
TEMPERATURE = 0.8
TOP_K = 50

In [ ]:
import os
import json
import time

PROGRESS_PATH = os.path.join(OUTPUT_DIR, f"generation_progress_{GENERATION_MODE}_encdec.json")

# Keep progress tied to this model and decoding settings.
import hashlib

def file_hash(path):
    digest = hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):
            digest.update(chunk)
    return digest.hexdigest()

identity = dict(model=file_hash(os.path.join(MODEL_OUTPUT_DIR,'model.pt')),
    tokenizer=file_hash(os.path.join(MODEL_OUTPUT_DIR,'tokenizer.json')),
    test=file_hash(test_file_path),mode=GENERATION_MODE,temperature=TEMPERATURE,
    top_k=TOP_K,budget=MAX_GENERATION_TOKENS,prompt_limit=generation_model.cfg.max_src_len,
    words=TARGET_GENERATION_WORDS,seed=SEED)
identity_path = PROGRESS_PATH+'.settings.json'
if os.path.isfile(PROGRESS_PATH):
    if not os.path.isfile(identity_path):
        raise ValueError('Choose a new progress filename for this model.')
    with open(identity_path) as f:
        if json.load(f) != identity:
            raise ValueError('Model/settings changed. Choose a new progress filename.')
else:
    with open(identity_path,'w') as f: json.dump(identity,f,indent=2)

# Resume predictions from this same model and decoding configuration.
if os.path.isfile(PROGRESS_PATH):
    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        predictions = json.load(f)
else:
    predictions = {}

# Check saved progress before resuming.


total = len(test_data)
start_time = time.time()

for example_id, record in test_data.items():
    if example_id in predictions:
        continue

    torch.manual_seed((SEED + int(hashlib.sha256(str(example_id).encode()).hexdigest()[:8],16)) % 2**31)
    continuation = generate_continuation(
        prompt=record["text"],
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=128,
        mode=GENERATION_MODE,
        temperature=TEMPERATURE,
        top_k=TOP_K,
        max_new_tokens=MAX_GENERATION_TOKENS,
    )


    predictions[example_id] = {"text": continuation}

    # Write atomically to preserve the previous progress file.
    temporary_path = PROGRESS_PATH + ".tmp"

    with open(temporary_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)

    os.replace(temporary_path, PROGRESS_PATH)

    print(
        f"Completed {len(predictions)}/{total} | "
        f"ID: {example_id} | "
        f"Elapsed: {(time.time() - start_time) / 60:.1f} min",
        flush=True,
    )


print("Generation loop complete.")

## 21. Save prediction.json

In [ ]:
if set(predictions) != set(test_data) or any(len(r['text'].split()) != TARGET_GENERATION_WORDS for r in predictions.values()):
    raise ValueError('Complete all 128-word predictions before saving the submission.')
import os
import json


PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("Saved:", PREDICTION_PATH)
print("Predictions:", len(predictions))

## 22. Create CodaBench Submission ZIP

In [ ]:
import zipfile

SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")

with zipfile.ZipFile(
    SUBMISSION_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    archive.write(
        PREDICTION_PATH,
        arcname="prediction.json",
    )

print('Submission ZIP saved:', SUBMISSION_ZIP)